# Supporter next-90-day donation prediction

Predict whether each supporter will donate in the next 90 days, using donation history up to a supporter-specific snapshot date.

- **Label**: `will_donate_90d` (binary)
- **Leakage avoidance**: snapshot date ensures features only use donations on/before snapshot; label uses donations after snapshot
- **Output**: `ml-outputs/supporter-next-donation/predictions.csv` (holdout predictions)

In [1]:
import os
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings('ignore')

DATA_DIR = '../lighthouse_csv_v7'
OUT_DIR = '../ml-outputs/supporter-next-donation'
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.25

HORIZON_DAYS = 90

print('Output folder:', OUT_DIR)

Output folder: ../ml-outputs/supporter-next-donation


In [2]:
supporters = pd.read_csv(os.path.join(DATA_DIR, 'supporters.csv'))
donations = pd.read_csv(os.path.join(DATA_DIR, 'donations.csv'))

# Dates
donations['donation_date'] = pd.to_datetime(donations['donation_date'], errors='coerce', utc=True)
supporters['created_at'] = pd.to_datetime(supporters['created_at'], errors='coerce', utc=True)

# Basic sanity
if 'supporter_id' not in supporters.columns:
    raise ValueError('Expected supporters.csv to have supporter_id')
if 'supporter_id' not in donations.columns:
    raise ValueError('Expected donations.csv to have supporter_id')

# Amount: treat monetary amount + estimated_value as signal, coerce to numeric
for c in ['amount', 'estimated_value']:
    if c in donations.columns:
        donations[c] = pd.to_numeric(donations[c], errors='coerce')

# Choose a horizon-safe global cutoff date so every supporter has a full label window
global_max = donations['donation_date'].max()
if pd.isna(global_max):
    raise ValueError('No valid donation_date values found')
latest_allowed_snapshot = global_max - pd.to_timedelta(HORIZON_DAYS, unit='D')

# Snapshot date per supporter: last donation on/before latest_allowed_snapshot
snap = (donations.dropna(subset=['donation_date'])
        .loc[donations['donation_date'] <= latest_allowed_snapshot]
        .groupby('supporter_id')['donation_date']
        .max()
        .rename('snapshot_date')
        .reset_index())

# Only supporters with at least one historical donation before snapshot
base = supporters.merge(snap, on='supporter_id', how='inner')

# Donations for features vs label
feat_d = donations.merge(base[['supporter_id', 'snapshot_date']], on='supporter_id', how='inner')
feat_d = feat_d[feat_d['donation_date'] <= feat_d['snapshot_date']].copy()

label_d = donations.merge(base[['supporter_id', 'snapshot_date']], on='supporter_id', how='inner')
label_d = label_d[(label_d['donation_date'] > label_d['snapshot_date']) &
                  (label_d['donation_date'] <= (label_d['snapshot_date'] + pd.to_timedelta(HORIZON_DAYS, unit='D')))].copy()

label_any = (label_d.groupby('supporter_id').size().rename('donations_in_window').reset_index())

core = base[['supporter_id', 'snapshot_date', 'supporter_type', 'relationship_type', 'region', 'country',
             'status', 'acquisition_channel', 'created_at']].copy()
core = core.merge(label_any, on='supporter_id', how='left')
core['donations_in_window'] = core['donations_in_window'].fillna(0).astype(int)
core['will_donate_90d'] = (core['donations_in_window'] > 0).astype(int)

print('Rows:', len(core))
print('Positive rate:', float(core['will_donate_90d'].mean()))
core[['supporter_id', 'snapshot_date', 'will_donate_90d']].head()

Rows: 59
Positive rate: 0.1864406779661017


,supporter_id,snapshot_date,will_donate_90d
0,1,2025-10-29 00:00:00+00:00,1
1,2,2025-05-08 00:00:00+00:00,0
2,3,2025-09-13 00:00:00+00:00,0
3,4,2025-11-24 00:00:00+00:00,0
4,5,2025-10-02 00:00:00+00:00,0


In [3]:
# Feature engineering from donation history up to snapshot

# Aggregate donation counts/values
agg = feat_d.groupby('supporter_id').agg(
    donation_count=('donation_id', 'count'),
    monetary_count=('donation_type', lambda s: int((s == 'Monetary').sum())),
    recurring_rate=('is_recurring', 'mean'),
    total_amount=('amount', 'sum'),
    median_amount=('amount', 'median'),
    total_estimated_value=('estimated_value', 'sum'),
    first_donation_date=('donation_date', 'min'),
    last_donation_date=('donation_date', 'max'),
).reset_index()

# Time-derived
agg['tenure_days'] = (core.set_index('supporter_id').loc[agg['supporter_id'], 'snapshot_date'].values - agg['first_donation_date'].values)
agg['tenure_days'] = pd.to_timedelta(agg['tenure_days']).dt.days.clip(lower=0)

snap_map = core.set_index('supporter_id')['snapshot_date']
agg['recency_days_at_snapshot'] = (snap_map.loc[agg['supporter_id']].values - agg['last_donation_date'].values)
agg['recency_days_at_snapshot'] = pd.to_timedelta(agg['recency_days_at_snapshot']).dt.days.clip(lower=0)

agg['donations_per_month_active'] = agg['donation_count'] / (1 + (agg['tenure_days'] / 30.4375))

# Mix features: donation_type and channel_source (top categories)
def _top_pivot(df: pd.DataFrame, key_col: str, value_col: str, top_n: int, prefix: str) -> pd.DataFrame:
    tmp = df[['supporter_id', key_col, value_col]].copy()
    tmp[key_col] = tmp[key_col].astype('string').fillna('Unknown')
    tmp[value_col] = pd.to_numeric(tmp[value_col], errors='coerce').fillna(0)

    top = tmp.groupby(key_col)[value_col].sum().sort_values(ascending=False).head(top_n).index.tolist()
    tmp[key_col] = tmp[key_col].where(tmp[key_col].isin(top), other='Other')

    piv = (tmp.pivot_table(index='supporter_id', columns=key_col, values=value_col, aggfunc='sum', fill_value=0)
             .reset_index())
    piv.columns = ['supporter_id'] + [f'{prefix}__{c}' for c in piv.columns[1:]]
    return piv

mix_amount = _top_pivot(feat_d, 'donation_type', 'amount', top_n=8, prefix='donation_type_amount')
mix_channel = _top_pivot(feat_d, 'channel_source', 'amount', top_n=10, prefix='channel_source_amount')

Xy = core.merge(agg, on='supporter_id', how='left')
Xy = Xy.merge(mix_amount, on='supporter_id', how='left')
Xy = Xy.merge(mix_channel, on='supporter_id', how='left')

# Fill NaNs created by merges
for c in Xy.columns:
    if c.startswith('donation_type_amount__') or c.startswith('channel_source_amount__'):
        Xy[c] = Xy[c].fillna(0)

Xy.head()

,supporter_id,snapshot_date,supporter_type,relationship_type,region,country,status,acquisition_channel,created_at,donations_in_window,...,donation_type_amount__InKind,donation_type_amount__Monetary,donation_type_amount__Skills,donation_type_amount__SocialMedia,donation_type_amount__Time,channel_source_amount__Campaign,channel_source_amount__Direct,channel_source_amount__Event,channel_source_amount__PartnerReferral,channel_source_amount__SocialMedia
0,1,2025-10-29 00:00:00+00:00,SocialMediaAdvocate,Local,Luzon,Philippines,Active,SocialMedia,2022-01-01 00:00:00+00:00,1,...,0.0,6878.12,0.0,0.0,0.0,0.00,4020.24,1040.91,0.00,1816.97
1,2,2025-05-08 00:00:00+00:00,Volunteer,Local,Mindanao,Philippines,Active,SocialMedia,2022-01-06 00:00:00+00:00,0,...,0.0,3480.08,0.0,0.0,0.0,0.00,2565.03,915.05,0.00,0.00
2,3,2025-09-13 00:00:00+00:00,MonetaryDonor,Local,Luzon,Philippines,Active,SocialMedia,2022-01-11 00:00:00+00:00,0,...,0.0,9225.71,0.0,0.0,0.0,3284.46,0.00,2292.88,3187.81,460.56
3,4,2025-11-24 00:00:00+00:00,MonetaryDonor,PartnerOrganization,Mindanao,Philippines,Active,Church,2022-01-16 00:00:00+00:00,0,...,0.0,8351.77,0.0,0.0,0.0,1017.09,2217.76,4290.93,0.00,825.99
4,5,2025-10-02 00:00:00+00:00,InKindDonor,PartnerOrganization,Mindanao,Philippines,Active,Website,2022-01-21 00:00:00+00:00,0,...,0.0,4738.58,0.0,0.0,0.0,3926.55,0.00,0.00,812.03,0.00


In [4]:
y = Xy['will_donate_90d'].astype(int)

id_like = ['supporter_id', 'snapshot_date', 'donations_in_window']
X = Xy.drop(columns=[c for c in (['will_donate_90d'] + id_like) if c in Xy.columns], errors='ignore')

# Drop high-leakage text fields
X = X.drop(columns=[c for c in ['email', 'phone', 'display_name', 'first_name', 'last_name', 'organization_name'] if c in X.columns], errors='ignore')

# Split
X_train, X_test, y_train, y_test, meta_train, meta_test = train_test_split(
    X, y,
    Xy[['supporter_id', 'snapshot_date']],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y if y.nunique() > 1 else None,
)

numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ],
    remainder='drop'
)

models = {
    'LogReg': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(
        n_estimators=500,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight='balanced_subsample',
        min_samples_leaf=2,
    ),
}

rows = []
fitters = {}
for name, model in models.items():
    pipe = Pipeline(steps=[('preprocess', preprocess), ('model', model)])
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    rows.append({
        'model': name,
        'roc_auc': roc_auc_score(y_test, proba) if y_test.nunique() > 1 else np.nan,
        'pr_auc': average_precision_score(y_test, proba) if y_test.nunique() > 1 else np.nan,
    })
    fitters[name] = pipe

results = pd.DataFrame(rows).sort_values(['pr_auc', 'roc_auc'], ascending=False)
results

,model,roc_auc,pr_auc
0,LogReg,0.555556,0.338828
1,RandomForest,0.416667,0.298701


In [5]:
best_name = results.iloc[0]['model']
best = fitters[best_name]

proba = best.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print('Best model:', best_name)
print('ROC AUC:', roc_auc_score(y_test, proba) if y_test.nunique() > 1 else 'n/a')
print('PR AUC:', average_precision_score(y_test, proba) if y_test.nunique() > 1 else 'n/a')
print('\nClassification report:')
print(classification_report(y_test, pred, digits=3))
print('Confusion matrix:\n', confusion_matrix(y_test, pred))

out = meta_test.copy()
out['y_true'] = y_test.values
out['p_will_donate_90d'] = proba
out['y_pred'] = pred

out_path = os.path.join(OUT_DIR, 'predictions.csv')
out.to_csv(out_path, index=False)
print('Wrote:', out_path)

out.sort_values('p_will_donate_90d', ascending=False).head(10)

Best model: LogReg
ROC AUC: 0.5555555555555556
PR AUC: 0.33882783882783885

Classification report:
              precision    recall  f1-score   support

           0      0.846     0.917     0.880        12
           1      0.500     0.333     0.400         3

    accuracy                          0.800        15
   macro avg      0.673     0.625     0.640        15
weighted avg      0.777     0.800     0.784        15

Confusion matrix:
 [[11  1]
 [ 2  1]]
Wrote: ../ml-outputs/supporter-next-donation/predictions.csv


,supporter_id,snapshot_date,y_true,p_will_donate_90d,y_pred
52,54,2025-11-23 00:00:00+00:00,0,0.874630,1
6,7,2025-11-24 00:00:00+00:00,1,0.841804,1
23,24,2025-10-14 00:00:00+00:00,0,0.283672,0
58,60,2025-05-29 00:00:00+00:00,0,0.217060,0
28,30,2025-07-29 00:00:00+00:00,0,0.096240,0
27,29,2025-10-08 00:00:00+00:00,0,0.091455,0
54,56,2025-11-12 00:00:00+00:00,1,0.087613,0
12,13,2025-04-07 00:00:00+00:00,0,0.079580,0
48,50,2025-07-08 00:00:00+00:00,0,0.075947,0
9,10,2025-11-09 00:00:00+00:00,0,0.074147,0


## Website surfacing + exported CSV schema

### Admin dashboard
- Use this as a **worklist prioritizer**: sort supporters by `p_will_donate_90d`.
- Display only for authorized staff (supporter-level predictions are sensitive).

### Public impact page (public-safe)
- Avoid showing supporter-level predictions. If desired, show an **aggregate forecast** (e.g., sum of predicted probabilities) without exposing individuals.

### Exported files
- `ml-outputs/supporter-next-donation/predictions.csv`
  - `supporter_id`, `snapshot_date`, `y_true`, `p_will_donate_90d`, `y_pred`